# Introduction
The dataset consists of numerical samples used for supervised learning.
Each row represents one observation, identified by a unique ID.

## Structure

### Training Set:
$$ID, INPUT_0, INPUT_1, ..., INPUT_{N-1}, TARGET_0, TARGET_1, TARGET_2, TARGET_3$$
Contains both input features and corresponding target values.

So $$X = (x_i) = (INPUT_0, INPUT_1, ..., INPUT_{N-1}) \quad i=1..500, N=12$$ and $$y_i=(TARGET_0, TARGET_1, TARGET_2, TARGET_3)\quad i=1..500$$

### Blind Test Set:
$$ID, INPUT_1, INPUT_2, ..., INPUT_N$$
Contains only input features; target values are omitted.

# 1. Training set analysis

In [2]:
## in the ML-CUP25-TR.csv comment is explained the shape of the dataset
n_inputs = 12

## Training set: ID, INPUTS, TARGET_1, TARGET_2, TARGET_3, TARGET_4 (last 4 columns)
columns = (
    ["ID"] +
    [f"INPUT_{i}" for i in range(n_inputs)] +
    [f"TARGET_{i}" for i in range(4)]
)

In [3]:
import pandas as pd
import numpy as np

ml_cup_tr = pd.read_csv("./data/MLC25/ML-CUP25-TR.csv", skiprows=7, names=columns)
ml_cup_tr.drop(columns=["ID"])

,INPUT_0,INPUT_1,INPUT_2,INPUT_3,INPUT_4,INPUT_5,INPUT_6,INPUT_7,INPUT_8,INPUT_9,INPUT_10,INPUT_11,TARGET_0,TARGET_1,TARGET_2,TARGET_3
0,-6.925642,-6.093158,-9.149763,-5.918488,4.391259,-1.059304,-5.031085,-6.932177,-5.805652,7.147028,4.555533,-5.694865,6.554997,10.688732,15.416160,-7.535628
1,-5.649870,-7.650998,-10.407383,-7.864047,3.790306,-1.673732,-8.493233,-8.143588,-9.447557,10.790796,6.266211,-5.551301,12.342252,-8.135250,23.787661,-3.270978
2,15.985886,14.192953,24.466835,12.551305,-7.788409,0.557977,23.145951,20.031774,14.516358,-21.024198,-10.410913,12.061133,28.542661,-14.132383,-56.408372,1.892238
3,12.774004,10.156462,18.588934,8.346695,-5.245173,-0.199274,14.500231,12.608063,12.411055,-15.479452,-8.871887,6.703585,20.253500,9.525402,-0.673842,40.295464
4,-4.019226,-4.043457,-5.095354,-3.147125,0.725466,-0.477673,-4.025913,-0.995364,-3.491760,3.385533,1.838361,-4.271710,-3.588910,6.050010,1.198489,-11.677909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,-7.843406,-7.094219,-14.356348,-6.721251,3.115052,-2.392241,-8.837664,-6.470696,-5.807141,9.754465,6.840160,-6.912569,14.093165,-4.845333,19.188406,-8.091055
496,15.792855,16.891394,27.492396,11.092835,-10.404277,1.038514,20.856860,20.624264,15.586675,-16.882227,-10.420515,10.612868,13.546922,27.986044,-7.274439,49.639830
497,13.200066,12.638262,26.459051,11.366576,-9.165648,1.587006,20.433137,20.044718,15.032366,-17.307257,-10.692552,10.053785,-11.158948,-27.368210,-22.150507,31.950889
498,-7.147499,-10.617766,-17.635645,-7.378656,5.586325,-3.656751,-7.872403,-9.290113,-8.199866,11.160446,5.605515,-6.487252,17.969044,2.482662,31.653539,-1.551031


In [ ]:
ml_cup_tr.shape

In [ ]:
ml_cup_tr.describe()

In [ ]:
X_tr = ml_cup_tr[[f"INPUT_{i}" for i in range(n_inputs)]].values
y_tr = ml_cup_tr[[f"TARGET_{i}" for i in range(4)]].values

In [ ]:
X_tr.shape

In [ ]:
y_tr.shape

In [ ]:
import seaborn as sbn

sbn.pairplot(ml_cup_tr, diag_kind="kde", kind="reg", corner=True, plot_kws={'line_kws':{'color':'red'}})

In [ ]:
sbn.heatmap(ml_cup_tr.corr(), cbar = False, annot = True, fmt=".1f")

## Preprocessing

The golden rule is
> - If your model is sensitive to feature magnitude (like neural networks, KNN, gradient boosting): 
>   - MinMaxScaler is usually best. 
> - If your model assumes normal distribution or uses distance metrics, 
>   - StandardScaler is usually better.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import *

In [ ]:
def plot_dataset_scaling(X, label):
    n_features = X.shape[1]
    for i in range(n_features):
        plt.hist(X[:, i], alpha=0.4, label=f"input_{i}")
    plt.legend(loc='upper right', ncol=3)
    if label:
        plt.title(label)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_dataset_scaling(X_tr, "Original TR")

### 1. MinMaxScaler
- Rescales features to a fixed range (usually 0–1 or -1–1). 
  - Preserves the shape of the original distribution but compresses values.
- Use when all features should have equal weight and you know the min/max bounds (e.g., neural networks).

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_tr)
X_tr_scaled = scaler.transform(X_tr)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "MinMax scaling TR")

In [ ]:
pd.DataFrame(X_tr).describe()

In [ ]:
pd.DataFrame(X_tr_scaled).describe()

Capiamo se ci sono tanti outlier utilizzando il Z-score, visto che ora abbiamo dati normalizzati.

In [ ]:
means = np.mean(X_tr_scaled, axis=0)
stds = np.std(X_tr_scaled, axis=0)

z_scores_minmax = (X_tr_scaled - means) / stds

In [ ]:
# Calcola il valore assoluto degli Z-score
abs_z_scores = np.abs(z_scores_minmax)

# Crea una maschera per i valori che superano la soglia di 3
outlier_mask = abs_z_scores > 3

# Conta quanti outlier sono presenti
numero_totale_outlier = np.sum(outlier_mask)
print(f"Outlier rilevati: {numero_totale_outlier}")

### 2. StandardScaler

- Centers data at mean 0 with unit variance (Z-score normalization). 
  - Keeps the relative shape, but shifts and scales by mean and standard deviation.
- Use when data may contain outliers or unknown range, or algorithms assume normality (e.g., linear/logistic regression, PCA, SVM).

In [ ]:
scaler = StandardScaler()
scaler.fit(X_tr)
X_tr_scaled = scaler.transform(X_tr)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "Standard scaling TR")

In [ ]:
pd.DataFrame(X_tr).describe()

In [ ]:
# apply(...) to suppress scientific notation, otherwise is confusonary
pd.DataFrame(X_tr_scaled).describe().apply(lambda s: s.apply(lambda x: format(x, 'g')))

# 2. Test set analysis

In [ ]:
## in the ML-CUP25-TS.csv comment is explained the shape of the dataset
n_inputs = 12

## Test set: ID, INPUTS
columns = (
    ["ID"] +
    [f"INPUT_{i}" for i in range(n_inputs)]
)

In [ ]:
import pandas as pd

ml_cup_ts = pd.read_csv("./data/MLC25/ML-CUP25-TS.csv", skiprows=7, names=columns)

In [ ]:
ml_cup_ts.describe()

In [ ]:
X_ts = ml_cup_ts[[f"INPUT_{i}" for i in range(n_inputs)]].values

In [ ]:
X_ts.shape

## Preprocessing

The golden rule is
> - If your model is sensitive to feature magnitude (like neural networks, KNN, gradient boosting): 
>   - MinMaxScaler is usually best. 
> - If your model assumes normal distribution or uses distance metrics, 
>   - StandardScaler is usually better.

In [ ]:
plot_dataset_scaling(X_ts, "Original TS")

## 1. MinMax Scaling

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_ts)
X_ts_scaled = scaler.transform(X_ts)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "MinMax scaling TS")

In [ ]:
pd.DataFrame(X_ts).describe()

In [ ]:
pd.DataFrame(X_ts_scaled).describe()

### 2. StandardScaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X_ts)
X_ts_scaled = scaler.transform(X_ts)

In [ ]:
plot_dataset_scaling(X_tr_scaled, "Standard scaling TS")

In [ ]:
pd.DataFrame(X_ts).describe()

In [ ]:
# apply(...) to suppress scientific notation, otherwise is confusonary
pd.DataFrame(X_tr_scaled).describe().apply(lambda s: s.apply(lambda x: format(x, 'g')))

### PCA: 1 component

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_tr_scaled)

In [ ]:
data = {"INPUT_0": X_pca[:, 0], 
        "TARGET_0": y_tr[:, 0], "TARGET_1": y_tr[:, 1], "TARGET_2": y_tr[:, 2], "TARGET_3": y_tr[:, 3]}
sbn.heatmap(
    pd.DataFrame(data=data).corr(), 
    cbar = False, 
    annot = True, 
    fmt=".1f"
    )

Bastano 2 componenti

In [ ]:
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')

In [ ]:
from sklearn.feature_selection import mutual_info_regression

def plot_mutual_information(X, y, feature_names, target_names, title="Mutual Information Scores"):
    """
    Calcola e visualizza la Mutual Information tra ogni feature e ogni target.
    """
    mi_matrix = pd.DataFrame(index=feature_names, columns=target_names)

    for col_y in target_names:
        # Calcolo della MI per il target specifico
        # random_state è importante per la riproducibilità
        mi_scores = mutual_info_regression(X, y[col_y], random_state=42)
        mi_matrix[col_y] = mi_scores

    plt.figure(figsize=(10, 6))
    sbn.heatmap(mi_matrix.astype(float), annot=True, fmt=".2f", cmap="YlGnBu")
    plt.title(title)
    plt.ylabel("Features / PCA Components")
    plt.xlabel("Targets")
    plt.show()
    
    return mi_matrix

# --- ESECUZIONE ---

# 1. MI sulle 12 feature originali (X_tr_scaled sono i tuoi dati originali scalati)
# Assumendo che X_tr_scaled sia un numpy array e y_tr pure
feature_cols = [f"INPUT_{i}" for i in range(12)]
target_cols = [f"TARGET_{i}" for i in range(4)]

print("Calcolo MI per le feature originali...")
mi_original = plot_mutual_information(
    X_tr_scaled, 
    pd.DataFrame(y_tr, columns=target_cols), 
    feature_cols, 
    target_cols,
    title="Mutual Information (Original Features)"
)

pca_names = [f"PCA_{i}" for i in range(X_pca.shape[1])]
print("Calcolo MI per le componenti PCA...")
mi_pca = plot_mutual_information(
    X_pca, 
    pd.DataFrame(y_tr, columns=target_cols), 
    pca_names, 
    target_cols,
    title="Mutual Information (PCA Components)"
)

### PCA: 2 components

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tr_scaled)

data = {"INPUT_0": X_pca[:, 0], 
        "INPUT_1": X_pca[:, 1], 
        "TARGET_0": y_tr[:, 0], "TARGET_1": y_tr[:, 1], "TARGET_2": y_tr[:, 2], "TARGET_3": y_tr[:, 3]}
pca_names = [f"PCA_{i}" for i in range(X_pca.shape[1])]

In [ ]:
sbn.heatmap(
    pd.DataFrame(data=data).corr(), 
    cbar = False, 
    annot = True, 
    fmt=".1f"
    )

In [ ]:
print("Calcolo MI per le componenti PCA...")
mi_pca = plot_mutual_information(
    X_pca, 
    pd.DataFrame(y_tr, columns=target_cols), 
    pca_names, 
    target_cols,
    title="Mutual Information (PCA Components)"
)

### PCA: 4 components

In [ ]:
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_tr_scaled)

data = {"INPUT_0": X_pca[:, 0], 
        "INPUT_1": X_pca[:, 1], 
        "INPUT_2": X_pca[:, 2], 
        "INPUT_3": X_pca[:, 3], 
        "TARGET_0": y_tr[:, 0], "TARGET_1": y_tr[:, 1], "TARGET_2": y_tr[:, 2], "TARGET_3": y_tr[:, 3]}
pca_names = [f"PCA_{i}" for i in range(X_pca.shape[1])]

In [ ]:
sbn.heatmap(
    pd.DataFrame(data=data).corr(), 
    cbar = False, 
    annot = True, 
    fmt=".1f"
    )

In [ ]:
print("Calcolo MI per le componenti PCA...")
mi_pca = plot_mutual_information(
    X_pca, 
    pd.DataFrame(y_tr, columns=target_cols), 
    pca_names, 
    target_cols,
    title="Mutual Information (PCA Components)"
)